In [2]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np   

In [3]:
## Initialize the model
model=SentenceTransformer('all-MiniLM-L6-v2')

## Sample text
text="""
LangChain is a framework for building applications with LLMs.
Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.
You can create chains, agents, memory, and retrievers. 
The Eiffel Tower is located in Paris.
France is a popular tourist destination.
LangChain treats prompts as first class citizens. You can template them, version them, and swap parts in and out without rewriting everythin
"""

## Step 1 : Split into sentences
sentences=[s.strip() for s in text.split("\n") if s.strip()]
sentences

## Step 2 : Embedding each sentence
embeddings = model.encode(sentences)

## Step 3: Initialize parameters

threshold = 0.7 # control chunk tightness
chunks =[]
current_chunk = [sentences[0]]

## Step 4: Semantic grouping based on threshold  

for i in range(1, len(sentences)):
    sim = cosine_similarity(
        [embeddings[i - 1]],
        [embeddings[i]]
    )[0][0]

    if sim>=threshold:
        current_chunk.append(sentences[i])
    else:
        chunks.append(" ".join(current_chunk))
        current_chunk=[sentences[i]]

# Append the last chunk
chunks.append(" ".join(current_chunk))

# Output the chunks
print("\n📌 Semantic Chunks:")
for idx, chunk in enumerate(chunks):
    print(f"\nChunk {idx+1}:\n{chunk}")


📌 Semantic Chunks:

Chunk 1:
LangChain is a framework for building applications with LLMs. Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.

Chunk 2:
You can create chains, agents, memory, and retrievers.

Chunk 3:
The Eiffel Tower is located in Paris.

Chunk 4:
France is a popular tourist destination.

Chunk 5:
LangChain treats prompts as first class citizens. You can template them, version them, and swap parts in and out without rewriting everythin


### Issues / improvements (important)

**1. You’re comparing only adjacent sentences**

* sim( sentence[i-1], sentence[i] ) can break when a topic continues but one sentence is a “bridge” or short transition.

* Better: compare the new sentence to the chunk centroid (mean embedding of the current chunk) or at least the first sentence of the chunk.

**2. Sentence splitting is too weak**

* text.split("\n") only splits on new lines, not real sentences.

* In real text, many sentences sit on the same line. You’ll end up embedding huge lines and chunking becomes noisy.

* Better: use a sentence tokenizer (NLTK / spaCy) or a regex fallback.

**3. Potential crash on empty input**

* If text is empty or becomes empty after stripping, sentences[0] will throw an error.

**4. Threshold value isn’t universal**

* 0.7 might be okay for your toy example, but on other data it may over-split or under-split.

* Better: tune threshold using distribution (percentile) or make it adaptive.

**5. No chunk size limits**

* In RAG, you usually want a max token/char limit. Otherwise one “similar” topic can grow too big and hurt retrieval.

**6. Missing imports (in the snippet)**

* You used cosine_similarity but didn’t show imports. Make sure you have:

In [4]:
import re
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# 1) Model
model = SentenceTransformer("all-MiniLM-L6-v2")

text = """
LangChain is a framework for building applications with LLMs.
Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.
You can create chains, agents, memory, and retrievers. 
The Eiffel Tower is located in Paris.
France is a popular tourist destination.
LangChain treats prompts as first class citizens. You can template them, version them, and swap parts in and out without rewriting everything.
"""

# 2) Sentence split (regex fallback)
def split_sentences(t: str):
    t = t.strip()
    if not t:
        return []
    # Split on sentence endings OR newlines, keep it simple
    parts = re.split(r'(?<=[.!?])\s+|\n+', t)
    return [p.strip() for p in parts if p.strip()]

sentences = split_sentences(text)
if not sentences:
    print("No text to chunk.")
    raise SystemExit 

# 3) Embed
embeddings = model.encode(sentences, normalize_embeddings=True)

threshold = 0.65        # start here; tune per your data
max_chars = 450         # safeguard chunk size
chunks = []

current_chunk = [sentences[0]]
current_embs = [embeddings[0]]

def chunk_text_len(chunk_list):
    return len(" ".join(chunk_list))

for i in range(1, len(sentences)):
    # centroid embedding of current chunk
    centroid = np.mean(current_embs, axis=0, keepdims=True)
    sim = cosine_similarity(centroid, embeddings[i].reshape(1, -1))[0][0]

    # if similar AND chunk isn't too large, keep adding
    if sim >= threshold and (chunk_text_len(current_chunk) + 1 + len(sentences[i]) <= max_chars):
        current_chunk.append(sentences[i])
        current_embs.append(embeddings[i])
    else:
        chunks.append(" ".join(current_chunk))
        current_chunk = [sentences[i]]
        current_embs = [embeddings[i]]

chunks.append(" ".join(current_chunk))

print("\n📌 Semantic Chunks:")
for idx, chunk in enumerate(chunks, 1):
    print(f"\nChunk {idx}:\n{chunk}")



📌 Semantic Chunks:

Chunk 1:
LangChain is a framework for building applications with LLMs. Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.

Chunk 2:
You can create chains, agents, memory, and retrievers.

Chunk 3:
The Eiffel Tower is located in Paris.

Chunk 4:
France is a popular tourist destination.

Chunk 5:
LangChain treats prompts as first class citizens.

Chunk 6:
You can template them, version them, and swap parts in and out without rewriting everything.


## RAG Example


In [5]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
# --- Core imports ---
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableMap, RunnableLambda, RunnablePassthrough

# Vector store + embeddings
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS 
from langchain.chat_models import init_chat_model 
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")  


In [6]:
import re
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# If you're using LangChain Documents:
from langchain_core.documents import Document


class ThresholdSemanticChunker:
    """Semantic sentence chunker using centroid similarity + threshold."""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2", threshold: float = 0.7, max_chars: int = 450):
        self.model = SentenceTransformer(model_name)
        self.threshold = float(threshold)
        self.max_chars = int(max_chars)

    @staticmethod
    def split_sentences(text: str) -> list[str]:
        text = (text or "").strip()
        if not text:
            return []
        parts = re.split(r'(?<=[.!?])\s+|\n+', text)
        return [p.strip() for p in parts if p.strip()]

    @staticmethod
    def chunk_text_len(chunk_list: list[str]) -> int:
        return len(" ".join(chunk_list))

    def split_text_chunks(self, text: str) -> list[str]:
        """Split a single text string into semantic chunks (strings)."""
        sentences = self.split_sentences(text)
        if not sentences:
            return []

        embeddings = self.model.encode(sentences, normalize_embeddings=True)

        chunks = []
        current_chunk = [sentences[0]]
        current_embs = [embeddings[0]]

        for i in range(1, len(sentences)):
            centroid = np.mean(current_embs, axis=0, keepdims=True)
            sim = cosine_similarity(centroid, embeddings[i].reshape(1, -1))[0][0]

            next_len = self.chunk_text_len(current_chunk) + 1 + len(sentences[i])

            if sim >= self.threshold and next_len <= self.max_chars:
                current_chunk.append(sentences[i])
                current_embs.append(embeddings[i])
            else:
                chunks.append(" ".join(current_chunk))
                current_chunk = [sentences[i]]
                current_embs = [embeddings[i]]

        chunks.append(" ".join(current_chunk))
        return chunks

    def split_documents(self, docs: list[Document]) -> list[Document]:
        """Split a list of LangChain Documents into smaller chunked Documents."""
        result = []
        for doc in docs:
            for chunk in self.split_text_chunks(doc.page_content):
                result.append(Document(page_content=chunk, metadata=doc.metadata))
        return result


In [7]:
text = """LangChain is a framework designed to help developers build applications powered by large language models.
It provides abstractions that simplify prompt management, memory handling, and tool integration.
Developers can combine LLMs with vector stores, retrievers, and external APIs to build intelligent systems.
LangChain treats prompts as first-class citizens and allows versioning and reuse across applications.

Semantic chunking is an important technique when preparing data for retrieval augmented generation systems.
Instead of splitting text by fixed character length, semantic chunking groups sentences based on meaning.
This improves retrieval accuracy because each chunk represents a single coherent idea.

The Eiffel Tower is located in Paris and was completed in 1889.
It is one of the most visited monuments in the world.
France is known for its culture, food, and historical landmarks."""

doc = Document(page_content=text, metadata={"source": "sample"})
chunker = ThresholdSemanticChunker(threshold=0.68)

chunks = chunker.split_documents([doc])   # ✅ returns list[Document]

In [8]:
def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

In [ ]:
chunker = ThresholdSemanticChunker(threshold=0.68)
chunks = chunker.split_documents([doc]) 
chunks

load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "").strip()
embeddings = OpenAIEmbeddings(model="text-embedding-3-small", openai_api_key=OPENAI_API_KEY)
embeddings
print(embeddings.embed_query("hello world")[:5])   # should return a list of floats

# -------------------- 2) Vector store --------------------
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})


[-0.00676333112642169, -0.03919631987810135, 0.034175805747509, 0.02876211516559124, -0.02478501945734024]


In [31]:
from dotenv import load_dotenv
import os
load_dotenv()

from langchain_groq import ChatGroq

GROQ_API_KEY = os.getenv("GROQ_API_KEY", "").strip()
print("GROQ key loaded:", bool(GROQ_API_KEY), "length:", len(GROQ_API_KEY))


# llm = init_chat_model("groq:meta-llama/llama-4-maverick-17b-128e-instruct", temperature=0)

# # llm = ChatGroq(
# #     model="meta-llama-4-maverick-17b-128e-instruct",  # see note below
# #     temperature=0,
# #     groq_api_key=GROQ_API_KEY,
# # )

# # sanity test BEFORE RAG
# print(llm.invoke("Say only OK").content)


GROQ key loaded: True length: 37


In [ ]:
# from langchain_community.embeddings import HuggingFaceEmbeddings
# from langchain_community.vectorstores import FAISS

# embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
# vectorstore = FAISS.from_documents(chunks, embeddings)

# retriever = vectorstore.as_retriever(search_kwargs={"k": 4})  # k controls how many chunks to retrieve


In [18]:
retriever.invoke("What is langchain ?")

[Document(id='5e68e1b5-2a5f-4c7a-ba78-957f6e52d88a', metadata={'source': 'sample'}, page_content='LangChain is a framework designed to help developers build applications powered by large language models.'),
 Document(id='755ab7d7-17dc-4deb-a19e-d02641021004', metadata={'source': 'sample'}, page_content='LangChain treats prompts as first-class citizens and allows versioning and reuse across applications.'),
 Document(id='05afabf6-760c-4492-91ab-8871d4588893', metadata={'source': 'sample'}, page_content='France is known for its culture, food, and historical landmarks.'),
 Document(id='9db2ffbd-2087-4dad-8086-7cc79d4214e1', metadata={'source': 'sample'}, page_content='Instead of splitting text by fixed character length, semantic chunking groups sentences based on meaning.')]

In [32]:
# -------------------- 3) Prompt --------------------
template = """Answer the question based on the following context:

{context}

Question: {question}
"""
prompt = PromptTemplate.from_template(template)

In [ ]:
# -------------------- 4) LLM --------------------


from langchain_openai import ChatOpenAI
llm = ChatOpenAI(
    temperature=0,
    model_name="gpt-3.5-turbo"
)
# sanity test BEFORE RAG
print(llm.invoke("Say only OK").content)

# -------------------- 5) LCEL RAG chain --------------------

rag_chain = (
    RunnableMap(
        {
            "context": lambda x: retriever.invoke(x["question"]),
            "question": lambda x: x["question"],
        }
    )
    | RunnableLambda(lambda x: {"context": format_docs(x["context"]), "question": x["question"]})
    | prompt
    | llm
    | StrOutputParser()
)


OK


In [42]:
# Run Query 

# -------------------- 6) Run --------------------
rag_chain.invoke({"question": "Where is the Eiffel Tower located?"})



'The Eiffel Tower is located in Paris, France.'

### Chain explanation 

✅ Step 1: RunnableMap({...})
This runs two functions in parallel, producing a new dict.

1A) Build "context"

lambda x: retriever.invoke(x["question"])

* Takes your dict x
* Pulls out x["question"] (a string)
* Calls retriever with that string
* Returns: list[Document]


1B) Build "question"

lambda x: x["question"]

* Just passes the question string through

{
  "context": [Document(...), Document(...), ...],   # list of docs
  "question": "Where is the Eiffel Tower located?"
}


✅ Step 2: RunnableLambda(...) formatting step

lambda x: {"context": format_docs(x["context"]), "question": x["question"]}

* Takes "context" which is list of docs
* Converts it to a single string using format_docs
* Keeps "question" the same

✅ Output becomes:


{
  "context": "Doc1 text...\n\nDoc2 text...\n\nDoc3 text...",
  "question": "Where is the Eiffel Tower located?"
}


✅ Step 3: prompt

The prompt template fills {context} and {question}.

✅ Output: a full prompt like:



Answer the question based on the following context:

Doc1...
Doc2...

Question: Where is the Eiffel Tower located?


✅ Step 4: llm

The model reads the prompt and generates the answer.


✅ Step 5: StrOutputParser()

Extracts plain text from the AI message.

✅ Final output:


"The Eiffel Tower is located in Paris, France."






In [44]:
# -------------------- 5) LCEL RAG chain --------------------
rag_chain = (
    {
        "context": retriever | RunnableLambda(format_docs),
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [45]:
answer = rag_chain.invoke("Where is the Eiffel Tower located?")
print(answer)
 

The Eiffel Tower is located in Paris, France.


### Explanation

✅ Step 1: The dict mapping { "context": ..., "question": ... }

This also runs two branches.

1A) "question": RunnablePassthrough()

RunnablePassthrough() means: “don’t change it, just pass it through”.

So "question" becomes:

"Where is the Eiffel Tower located?"



1B) "context": retriever | RunnableLambda(format_docs)

This is a mini-pipeline (a sub-chain):

(a) retriever

Receives the input string

Returns list[Document]

(b) RunnableLambda(format_docs)

Receives that list of docs

Joins them into one string

So "context" becomes:

"Doc1 text...\n\nDoc2 text...\n\nDoc3 text..."


✅ Output after this mapping becomes:


{
  "context": "Doc1...\n\nDoc2...\n\nDoc3...",
  "question": "Where is the Eiffel Tower located?"
}

✅ Step 2: prompt

Same as code 1: formats the final prompt.

✅ Step 3: llm

Same: generates the response.


✅ Step 4: StrOutputParser()

Same: returns plain string.



### So what’s the real difference between Code 1 and Code 2?

**Input style**

* Code 1 expects a dict: {"question": ...}
* Code 2 expects a string: "..." (or dict if you redesign it)

**Composition style**

* Code 1 does retrieval inside lambdas, then formats in a second lambda.
* Code 2 builds the pipeline in a more LCEL-native way using | composition.

**Practical difference**

* Code 2 is usually cleaner, easier to extend, and less error-prone.
* Code 1 is fine, but you must always pass dict input with "question".



**Tiny mental model to remember 🧠**

* RunnableMap({...}) or {...} mapping = “build a dict of inputs for the next step”
* RunnablePassthrough() = “keep the original input”
* retriever | format_docs = “retrieve docs then convert to text”
* | prompt | llm | parser = “stuff it into a prompt, ask model, return string”